# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. `5`, форма массива `(2, 3)`: элемент второй строки с индексом 2 равен 5.
2. `1 8`: максимум 8 впервые встречается на позиции 1.
3. `99`: срез `a[::3]` — это представление исходного массива, поэтому изменение `b[0]` меняет `a[0]`.
4. `2 0.5`: два элемента больше 3; среднее булевой маски равно доле `True`.
5. `(2, 3) 0.25`: `zeros` создаёт массив формы `(2, 3)`, второй элемент `linspace(0, 1, 5)` равен 0.25.


In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [3]:
i_max = np.argmax(amount)
print("1) Самый дорогой заказ")
print("Позиция:", i_max)
print("Сумма:", f"{amount[i_max]:,.2f} ₸")
print("Клиент:", cust[i_max])
print("Город:", city[i_max])
print("Комментарий: это заказ с максимальной суммой среди всех строк.")

revenue_paid = amount[paid].sum()
amount_unpaid = amount[~paid].sum()
print("\\n2) Оплаченные и неоплаченные заказы")
print("Выручка оплаченных:", f"{revenue_paid:,.2f} ₸")
print("Сумма неоплаченных:", f"{amount_unpaid:,.2f} ₸")
print("Комментарий: в выручку включены только заказы со статусом «оплачен».")

shymkent_mask = city == "Шымкент"
shymkent_count = shymkent_mask.sum()
shymkent_share = shymkent_count / len(city)
print("\\n3) Заказы в Шымкенте")
print("Количество:", shymkent_count)
print("Доля:", f"{shymkent_share:.2%}")
print("Комментарий: доля рассчитана от общего количества заказов.")


1) Самый дорогой заказ
Позиция: 1001
Сумма: 9,999,990.00 ₸
Клиент: TEST
Город: Алматы
Комментарий: это заказ с максимальной суммой среди всех строк.
\n2) Оплаченные и неоплаченные заказы
Выручка оплаченных: 101,909,360.00 ₸
Сумма неоплаченных: 7,974,000.00 ₸
Комментарий: в выручку включены только заказы со статусом «оплачен».
\n3) Заказы в Шымкенте
Количество: 804
Доля: 14.72%
Комментарий: доля рассчитана от общего количества заказов.


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [4]:
discount_mask = paid & (amount > 50_000)
loss_all = amount[discount_mask].sum() * 0.15

discount_mask_no_test = discount_mask & (cust != "TEST")
loss_no_test = amount[discount_mask_no_test].sum() * 0.15

print("4) Потери выручки из-за скидки 15%")
print("Со всеми заказами:", f"{loss_all:,.2f} ₸")
print("Без клиента TEST:", f"{loss_no_test:,.2f} ₸")
print("Разница:", f"{loss_all - loss_no_test:,.2f} ₸")
print("Комментарий: TEST может сильно увеличить расчёт, если у него есть аномально дорогие заказы.")

print("\\n5) Доставка за 1–2 дня по городам")
print("Город | Быстрых доставок | Всего заказов | Доля")
for c in np.unique(city):
    city_mask = city == c
    fast_mask = city_mask & np.isfinite(days) & ((days == 1) | (days == 2))
    count_fast = fast_mask.sum()
    count_all = city_mask.sum()
    share = count_fast / count_all if count_all else np.nan
    print(f"{c} | {count_fast} | {count_all} | {share:.2%}")

avg_paid = amount[paid].mean()
print("\\n6) Средний чек")
print("Средний чек оплаченного заказа:", f"{avg_paid:,.2f} ₸")
print("Комментарий: средний чек считаем только по оплаченным заказам, без округления до целого.")


4) Потери выручки из-за скидки 15%
Со всеми заказами: 5,702,422.50 ₸
Без клиента TEST: 1,202,427.00 ₸
Разница: 4,499,995.50 ₸
Комментарий: TEST может сильно увеличить расчёт, если у него есть аномально дорогие заказы.
\n5) Доставка за 1–2 дня по городам
Город | Быстрых доставок | Всего заказов | Доля
Актобе | 247 | 397 | 62.22%
Алматы | 1412 | 2188 | 64.53%
Астана | 883 | 1390 | 63.53%
Атырау | 94 | 146 | 64.38%
Караганда | 365 | 538 | 67.84%
Шымкент | 240 | 804 | 29.85%
\n6) Средний чек
Средний чек оплаченного заказа: 20,780.86 ₸
Комментарий: средний чек считаем только по оплаченным заказам, без округления до целого.
